# Module 5 - Lab: Generate, Check and Store Data with phyphox

This 45-minute lab follows the early data pipeline from the Module 5 impulse: **generate, check, organise, store**.

The goal is to turn a fresh phyphox export into data that another person can find, understand, and reuse later.

## Timebox
- 10 min: Generate a measurement and export it.
- 10 min: Check whether the recording worked correctly.
- 15 min: Organise files, rename them, and separate raw from preprocessed data.
- 10 min: Store the artefact in hot storage and document the location.

## Module 5 ideas used in this lab
- A raw export is not yet research data.
- Check directly after generation, before analysis begins.
- The check asks whether the run recorded correctly, not whether the result is nice.
- Keep or delete failed runs deliberately.
- Raw data is sacred: never edit it in place.
- Use clear folders, naming conventions, and versions.
- Store hot data in Sciebo and document the path.

## Section 1: Import Libraries

Run this cell first. The notebook uses Python to inspect file formats, check exported data, and create a small storage checklist.

In [ ]:
# Section 1: Import Libraries
from src.lab_notebook_helpers import setup_lab_environment, select_recorded_data

globals().update(setup_lab_environment())

print('Libraries imported successfully.')

## Section 2: Generate the Measurement

Use phyphox on your phone to record one run on the model.

Before exporting, write down the context. This is the information that makes the file understandable later.

- What subsystem did you measure: `drivetrain` or `suspension`?
- Which quantity did you record: acceleration, light, rotation, etc.?
- Which phone/sensor did you use?
- What was the intended duration?
- Which run number is this?
- Did anything unusual happen during the run?

Export the phyphox data as CSV or Excel and place the raw export somewhere under `data/`.

## Section 3: Point to the Recorded Data

`metadata.json` stays compact. It records which file is used, which measurement type was performed, and a few naming/storage fields used later in the lab.

For the example data, switch between:

- `data/drivetrain/Example/Raw Data.csv`
- `data/drivetrain/Example/Raw Data.xlsx`
- `data/suspension/Example/Beschleunigung ohne g.xls`

In [ ]:
# Section 3: Point to the Recorded Data
recorded_data_path_override = None
measurement_type_override = None
save_metadata = False

# recorded_data_path_override = 'data/drivetrain/Example/Raw Data.csv'
# measurement_type_override = 'drivetrain'
# recorded_data_path_override = 'data/suspension/Example/Beschleunigung ohne g.xls'
# measurement_type_override = 'suspension'

globals().update(select_recorded_data(
    project_root,
    recorded_data_path_override=recorded_data_path_override,
    measurement_type_override=measurement_type_override,
    save_metadata=save_metadata,
))

print('Recorded data path:', selected_data_path)
print('File exists:', selected_data_path.exists())
display(metadata_summary)

## Section 4: Detect File Format and Load Data

One common data-quality problem is inconsistent format: comma vs. semicolon, decimal point vs. decimal comma. This section detects the format before loading.

Supported formats:
- Excel
- CSV comma, decimal point
- CSV tabulator, decimal point
- CSV semicolon, decimal point
- CSV tabulator, decimal comma
- CSV semicolon, decimal comma

In [ ]:
# Section 4: Detect File Format and Load Data
loaded_recorded_data = load_recorded_data(selected_data_path, project_root)
df_raw = loaded_recorded_data['table']
detected_format = loaded_recorded_data['format']

print('Detected format:', detected_format['format_label'])
print('Rows, columns:', df_raw.shape)
display(df_raw.head())
display(pd.DataFrame([summarize_loaded_data(loaded_recorded_data)]).T.rename(columns={0: 'value'}))

## Section 5: Inspect Metadata Stored with the Data

Use this section as a quick metadata dashboard. Check whether the file, device, sensor, time information, and measurement type match the run you intended to record.

Questions to answer:

- Which device and sensor created this file?
- When did recording start and stop?
- Does the metadata match `metadata.json`?
- What information would another group still need?

In [ ]:
# Section 5: Inspect Metadata Stored with the Data
recorded_metadata = loaded_recorded_data['recording_metadata']
metadata_overview = prepare_recording_metadata_overview(loaded_recorded_data, metadata)

print('Recorded data overview')
display(metadata_overview['overview'])

if 'metadata_files' in metadata_overview and not metadata_overview['metadata_files'].empty:
    print('Metadata files found')
    display(metadata_overview['metadata_files'])

if 'device_metadata' in metadata_overview and not metadata_overview['device_metadata'].empty:
    print('Device and sensor metadata')
    display(metadata_overview['device_metadata'])

if 'time_metadata' in metadata_overview and not metadata_overview['time_metadata'].empty:
    print('Recording time metadata')
    display(metadata_overview['time_metadata'])

if 'excel_sheets' in metadata_overview and not metadata_overview['excel_sheets'].empty:
    print('Excel workbook metadata')
    display(metadata_overview['excel_sheets'])

## Section 6: Check Correct Recording

This is a **recording check**, not an analysis result.

First, make a rough plot so obvious problems become visible immediately. Then use the check table to decide whether the run was recorded correctly.

Ask the Module 5 questions:

- Did the file actually write?
- Is the duration plausible?
- Are the expected columns and axes present?
- Are there missing values, flat lines, clipping, or dropouts?
- Does the plot look plausible for `drivetrain` or `suspension`?
- Did the generation run as set up?

In [ ]:
# Section 6: Check Correct Recording
quality_report, time_column = create_recording_quality_report(df_raw, selected_data_path, metadata)

print('Rough first visualisation')
plot_first_measurement_overview(df_raw, metadata, time_column)

print('Recording quality checks')
display(quality_report)

print('Numeric summary')
display(df_raw.describe(include='all').T)

## Section 7: Keep or Delete Decision

The workflow stops here until you decide what happens to this recording.

- Choose **Keep** if the recording is usable for storage and later analysis.
- Choose **Delete** if the recording should be repeated.
- If you choose **Delete**, write a short reason. The reason is documented in `data/lab5_keep_delete_decisions.jsonl`.

Only continue with Section 8 after the notebook confirms the **Keep** decision.


In [ ]:
# Section 7: Keep or Delete Decision
keep_delete_decision = display_keep_delete_decision(
    project_root,
    selected_data_path,
    metadata,
    quality_report=quality_report,
)


In [ ]:
# Section 7: Continue only after Keep
require_keep_decision(keep_delete_decision)


## Section 8: Organise Raw, Preprocessed, Analysed, Docs

Use a shallow structure that another person can navigate.

Recommended structure:

```text
data/<subsystem>/<run_id>/
  raw/
  preprocessed/
  analysed/
  docs/
```

Raw data is sacred. Never edit it in place.

In [ ]:
# Section 8: Organise Raw, Preprocessed, Analysed, Docs
subsystem = metadata.get('measurement_type', 'drivetrain')
quantity = metadata.get('quantity', 'measurement')
run_name = metadata.get('run_name', 'Example')
run_number = 1
date_iso = '2026-07-01'
project = 'rdm'
stage = metadata.get('data_stage', 'raw')
version = metadata.get('version', 'v0.1.0')

run_id = f'{date_iso}_{project}_{subsystem}_{quantity}_run{run_number:02d}'
target_root = project_root / 'data' / subsystem / run_id
raw_folder = target_root / 'raw'
preprocessed_folder = target_root / 'preprocessed'
analysed_folder = target_root / 'analysed'
docs_folder = target_root / 'docs'

for folder in [raw_folder, preprocessed_folder, analysed_folder, docs_folder]:
    folder.mkdir(parents=True, exist_ok=True)

recommended_filename = f'{run_id}_{stage}_{version}{selected_data_path.suffix.lower()}'
target_raw_path = raw_folder / recommended_filename

print('Run name:', run_name)
print('Run ID:', run_id)
print('Recommended raw filename:', recommended_filename)
print('Target raw path:', target_raw_path)

## Section 9: Copy Raw Export Without Editing It

This creates a copy of the raw export with a clear name. The original export is not modified.

Only set `copy_raw_file = True` after checking the target path.

In [ ]:
# Section 9: Copy Raw Export Without Editing It
copy_raw_file = False

if copy_raw_file:
    if target_raw_path.exists():
        raise FileExistsError(f'Target file already exists. Choose a new version: {target_raw_path}')
    shutil.copy2(selected_data_path, target_raw_path)
    print('Copied raw export to:', target_raw_path)
else:
    print('Dry run only. Set copy_raw_file = True to copy the raw export.')

## Section 10: Create a Documentation File

A README explains the rules in plain words. This one records what was checked, where the data came from, and what storage path should be documented in the DMP.

In [ ]:
# Section 10: Create a Documentation File
sciebo_path = metadata.get('hot_storage_path') or 'sciebo://CHANGE_ME/project/path/to/hot-storage'

lab5_summary = {
    'recorded_data_path': str(selected_data_path.relative_to(project_root)).replace('\\', '/'),
    'measurement_type': subsystem,
    'run_name': run_name,
    'quantity': quantity,
    'data_stage': stage,
    'version': version,
    'detected_format': detected_format,
    'metadata_source': recorded_metadata.get('source'),
    'recommended_run_id': run_id,
    'recommended_raw_path': str(target_raw_path.relative_to(project_root)).replace('\\', '/'),
    'row_count': int(df_raw.shape[0]),
    'column_count': int(df_raw.shape[1]),
    'columns': df_raw.columns.tolist(),
    'quality_checks': quality_report.to_dict(orient='records'),
    'hot_storage_path': sciebo_path,
    'backup_note': 'Hot data should be synced, backed up, shared with the team, and documented in the DMP.'
}

readme_text = f'''# {run_id}

## What was measured
- Measurement type: {subsystem}
- Quantity: {quantity}
- Run name: {run_name}
- Run number: {run_number}
- Source file: {lab5_summary['recorded_data_path']}
- Detected format: {detected_format['format_label']}

## Data handling
- Raw data is stored separately and must not be edited in place.
- Preprocessed and analysed files are new artefacts and need their own names and versions.
- File names use ISO date, project, subsystem, quantity, run number, stage, and version.

## Hot storage
- Sciebo path: {sciebo_path}
- Document this path in the DMP.

## Recording check
{quality_report.to_string(index=False)}
'''

summary_path = docs_folder / f'{run_id}_lab5_summary.json'
readme_path = docs_folder / 'README.md'

write_docs = False
if write_docs:
    with summary_path.open('w', encoding='utf-8') as f:
        json.dump(lab5_summary, f, indent=2)
    readme_path.write_text(readme_text, encoding='utf-8')
    print('Wrote summary:', summary_path)
    print('Wrote README:', readme_path)
else:
    print('Dry run only. Set write_docs = True to write documentation files.')
    print(readme_text[:1200])

## Section 11: Hot Storage Checklist

Use Sciebo as hot storage for active working data.

- Synced and backed up: a laptop failure must not cost a run.
- Shared with the team: no email attachments as the main data store.
- Path documented: the location goes into the DMP.
- 3-2-1 thinking: three copies, two storage types, one elsewhere. In this course, Sciebo covers the backup side for hot data.

**Sciebo path documented in DMP:** yes / no

**Who can access the folder:**

**Remaining action before analysis:**

## Section 12: Final Lab Check

Before leaving Lab 5, make sure you can answer these questions:

- Did the run record correctly?
- Did you keep, repeat, or delete it for a documented reason?
- Is raw data separate from preprocessed or analysed data?
- Does the filename follow the agreed naming convention?
- Is the version explicit, for example `v0.1.0`?
- Is the Sciebo hot-storage path documented?
- Can another group find this file tomorrow and understand what it is?

The stored artefact is now ready to become input for the following analysis modules.